In [ ]:
# ==========================================
# MODELING
# ==========================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    average_precision_score,
    f1_score
)

from imblearn.over_sampling import SMOTE

# -----------------------------------
# Load Data
# -----------------------------------

df = pd.read_csv(
    "../data/processed/fraud_processed.csv"
)

print("Dataset Shape:", df.shape)

# -----------------------------------
# Drop Unnecessary Columns
# -----------------------------------

columns_to_drop = [
    "signup_time",
    "purchase_time",
    "user_id",
    "device_id",
    "lower_bound_ip_address",
    "upper_bound_ip_address"
]

existing_cols = [
    col for col in columns_to_drop
    if col in df.columns
]

df.drop(columns=existing_cols, inplace=True)

print("Shape after dropping columns:", df.shape)

# -----------------------------------
# One-Hot Encoding
# -----------------------------------

categorical_cols = [
    "source",
    "browser",
    "sex",
    "country"
]

existing_cats = [
    col for col in categorical_cols
    if col in df.columns
]

df = pd.get_dummies(
    df,
    columns=existing_cats,
    drop_first=True
)

print("Shape after encoding:", df.shape)

# -----------------------------------
# Define Features and Target
# -----------------------------------

X = df.drop("class", axis=1)
y = df["class"]

print("X Shape:", X.shape)
print("y Shape:", y.shape)

# -----------------------------------
# Train Test Split
# -----------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    stratify=y,
    test_size=0.2,
    random_state=42
)

# -----------------------------------
# Scaling
# -----------------------------------

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# -----------------------------------
# SMOTE
# -----------------------------------

print("Applying SMOTE...")

smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train,
    y_train
)

print("\nBefore SMOTE:")
print(y_train.value_counts())

print("\nAfter SMOTE:")
print(y_train_smote.value_counts())

# ====================================
# Logistic Regression
# ====================================

print("\n" + "="*60)
print("LOGISTIC REGRESSION")
print("="*60)

lr = LogisticRegression(
    max_iter=1000
)

lr.fit(
    X_train_smote,
    y_train_smote
)

lr_pred = lr.predict(X_test)

print("\nConfusion Matrix")
print(confusion_matrix(y_test, lr_pred))

print("\nClassification Report")
print(classification_report(y_test, lr_pred))

lr_prob = lr.predict_proba(X_test)[:,1]

print(
    "\nAUC-PR:",
    average_precision_score(
        y_test,
        lr_prob
    )
)

print(
    "F1 Score:",
    f1_score(
        y_test,
        lr_pred
    )
)

# ====================================
# Random Forest
# ====================================

print("\n" + "="*60)
print("RANDOM FOREST")
print("="*60)

rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

rf.fit(
    X_train_smote,
    y_train_smote
)

rf_pred = rf.predict(X_test)

print("\nConfusion Matrix")
print(confusion_matrix(y_test, rf_pred))

print("\nClassification Report")
print(classification_report(y_test, rf_pred))

rf_prob = rf.predict_proba(X_test)[:,1]

print(
    "\nAUC-PR:",
    average_precision_score(
        y_test,
        rf_prob
    )
)

print(
    "F1 Score:",
    f1_score(
        y_test,
        rf_pred
    )
)

# ====================================
# Model Comparison
# ====================================

lr_f1 = f1_score(y_test, lr_pred)
rf_f1 = f1_score(y_test, rf_pred)

lr_auc = average_precision_score(
    y_test,
    lr_prob
)
rf_auc = average_precision_score(
    y_test,
    rf_prob
)

comparison = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest"
    ],
    "F1 Score": [
        lr_f1,
        rf_f1
    ],
    "AUC-PR": [
        lr_auc,
        rf_auc
    ]
})

print("\n")
print("="*60)
print("MODEL COMPARISON")
print("="*60)

print(comparison)

best_model = comparison.sort_values(
    by="F1 Score",
    ascending=False
).iloc[0]

print("\nBest Model:")
print(best_model["Model"])

print("\nModeling Completed Successfully")


Dataset Shape: (129146, 19)
Shape after dropping columns: (129146, 13)
Shape after encoding: (129146, 196)
X Shape: (129146, 195)
y Shape: (129146,)
Applying SMOTE...

Before SMOTE:
class
0    93502
1     9814
Name: count, dtype: int64

After SMOTE:
class
0    93502
1    93502
Name: count, dtype: int64

LOGISTIC REGRESSION

Confusion Matrix
[[21944  1432]
 [  709  1745]]

Classification Report
              precision    recall  f1-score   support

           0       0.97      0.94      0.95     23376
           1       0.55      0.71      0.62      2454

    accuracy                           0.92     25830
   macro avg       0.76      0.82      0.79     25830
weighted avg       0.93      0.92      0.92     25830


AUC-PR: 0.6794404246278134
F1 Score: 0.6197833422127509

RANDOM FOREST

Confusion Matrix
[[21835  1541]
 [  662  1792]]

Classification Report
              precision    recall  f1-score   support

           0       0.97      0.93      0.95     23376
           1       0.54